In [65]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from empiricaldist import Pmf, Cdf


########################## CARGA DESDE ARCHIVO LOCAL ##########################
# Cargar el archivo local
data = pd.read_csv("vista_agrosavia_Municipio_Funza_20260111.csv", sep=",", encoding="utf-8")

# Ver las primeras filas
display(data.head())





,Departamento,Municipio,Cultivo,Estado,Topografia,Drenaje,Riego,Fertilizantes aplicados,Secuencial
0,CUNDINAMARCA,FUNZA,Guisantes,Por establecer,Plano,Buen drenaje,Goteo,UREA,166.0
1,CUNDINAMARCA,FUNZA,Lechuga,Por establecer,Plano,Buen drenaje,Aspersión,15-15-15,248.0
2,CUNDINAMARCA,FUNZA,Zanahoria,Por establecer,Plano,Buen drenaje,Aspersión,15-15-15,249.0
3,CUNDINAMARCA,FUNZA,Maíz,Por establecer,Plano,Buen drenaje,Aspersión,15-15-15,306.0
4,CUNDINAMARCA,FUNZA,Lechuga,Establecido,Plano,Buen drenaje,Aspersión,"COMPOSTAJE+QUIMICO EDAFICO+(DAP,KCL+FERTILIZAC...",388.0


### Leyenda de las columnas del dataset 
  
dataset caracterización productiva / administrativa agrícola. (No de parámetros agro)  
Registro y seguimiento de unidades productivas agrícolas, enfocado en:  
qué se cultiva  
dónde  
en qué estado está el cultivo  
bajo qué condiciones generales (riego, drenaje, topografía)  
qué insumos se aplican  
  
Esto sirve para:  
planificación agrícola  
extensión rural  
diagnósticos productivos  
análisis de prácticas agrícolas  
políticas públicas / reportes institucionales  
  
⚠️ No sirve directamente para rendimiento, fertilidad o modelación agronómica fina, pero sí para análisis exploratorio, categórico y estadístico.  
⚠️ No hay variables continuas reales




Tabla 1: Descripción de las columnas del dataset
| Columna                 | Tipo                             | Comentario                    |
| ----------------------- | -------------------------------- | ----------------------------- |
| Departamento            | Categórica nominal               | No tiene orden                |
| Municipio               | Categórica nominal               | No tiene orden                |
| Cultivo                 | Categórica nominal               | No tiene orden                |
| Estado                  | Categórica **ordinal**           | Tiene orden lógico            |
| Topografia              | Categórica ordinal (débil)       | Plano < Ondulado < Escarpado  |
| Drenaje                 | Categórica ordinal               | Malo < Regular < Bueno        |
| Riego                   | Categórica nominal               | Tipo de riego                 |
| Fertilizantes aplicados | Categórica nominal (texto libre) | Alta cardinalidad             |
| Secuencial              | Numérica discreta                | **ID**, no variable analítica |



In [66]:
# Dimensión del dataset
filas = data.shape[0]  # Número de filas
columnas = data.shape[1]  # Número de columnas
print(f"El dataset tiene {filas} filas y {columnas} columnas.")
print("------"*4)
# Tipos de datos de cada columna
print("Tipos de datos de cada columna:")
display(data.dtypes)
print("------"*4)
# Contar valores nulos en cada columna
num_null = data.isnull().sum()
print("Número de valores nulos por columna:")
display(num_null)
print("------"*4)
# Contar valores únicos en cada columna.
print("Número de valores únicos por columna:")
display(data.nunique().sort_values())
print("------"*4)
# Trasformar columnas categóricas a tipo 'category' para optimizar memoria
for col in data.select_dtypes(include=['object']).columns:
    data[col] = data[col].astype('category')
# validar moda, max, min, media y mediana (descriptivas básicas solo de las 'categorías')
print("Estadísticas descriptivas para columnas categóricas:")
display(data.describe(include=['category']))



El dataset tiene 88 filas y 9 columnas.
------------------------
Tipos de datos de cada columna:


Departamento                object
Municipio                   object
Cultivo                     object
Estado                      object
Topografia                  object
Drenaje                     object
Riego                       object
Fertilizantes aplicados     object
Secuencial                 float64
dtype: object

------------------------
Número de valores nulos por columna:


Departamento               0
Municipio                  0
Cultivo                    0
Estado                     0
Topografia                 0
Drenaje                    0
Riego                      0
Fertilizantes aplicados    0
Secuencial                 0
dtype: int64

------------------------
Número de valores únicos por columna:


Departamento                1
Municipio                   1
Estado                      4
Drenaje                     4
Riego                       7
Topografia                  7
Fertilizantes aplicados    16
Cultivo                    27
Secuencial                 88
dtype: int64

------------------------
Estadísticas descriptivas para columnas categóricas:


,Departamento,Municipio,Cultivo,Estado,Topografia,Drenaje,Riego,Fertilizantes aplicados
count,88,88,88,88,88,88,88,88
unique,1,1,27,4,7,4,7,16
top,CUNDINAMARCA,FUNZA,Uchuva,Por establecer,Plano,Buen drenaje,Aspersión,No indica
freq,88,88,14,41,57,64,33,53


In [67]:
# Análisis por columnas categóricas - Cardinalidad y valores únicos por categoría
for col in data.select_dtypes(include=['category']).columns:
    print(f"\nColumna: {col}")
    print(f"Cardinalidad: {data[col].nunique()}")
    print("Valores únicos:")
    for val in data[col].cat.categories:
        print(f" - {val}") # Mostrar cada valor único en la categoría



Columna: Departamento
Cardinalidad: 1
Valores únicos:
 - CUNDINAMARCA

Columna: Municipio
Cardinalidad: 1
Valores únicos:
 - FUNZA

Columna: Cultivo
Cardinalidad: 27
Valores únicos:
 - Ajo
 - Alfalfa
 - Caducifolios
 - Café
 - Calas
 - Clavel
 - Fresa
 - Granadilla
 - Guisantes
 - Gulupa
 - Hipérico
 - Hortalizas Varias
 - Lechuga
 - Maracuyá
 - Maíz
 - No Indica
 - Papa de año
 - Pasto
 - Pasto Kikuyo
 - Pasto KinGrass
 - Pastos
 - Quinua
 - Ruscus
 - Tomate
 - Tomate de Arbol
 - Uchuva
 - Zanahoria

Columna: Estado
Cardinalidad: 4
Valores únicos:
 - ESTABLECIDO
 - Establecido
 - No indica
 - Por establecer

Columna: Topografia
Cardinalidad: 7
Valores únicos:
 - Moderadamente ondulado
 - No indica
 - Ondulado
 - Pendiente
 - Pendiente fuerte
 - Pendiente moderada
 - Plano

Columna: Drenaje
Cardinalidad: 4
Valores únicos:
 - Buen drenaje
 - Mal drenaje
 - No indica
 - Regular drenaje

Columna: Riego
Cardinalidad: 7
Valores únicos:
 - Aspersión
 - Cañon
 - Goteo
 - Manguera
 - No Indic

# Resumen del análisis del dataset

El dataset analizado corresponde a un registro categórico de prácticas agrícolas en una unidad territorial homogénea. Su estructura lo hace adecuado para análisis exploratorio descriptivo, distribuciones categóricas y pruebas de dependencia, pero no para modelación cuantitativa ni análisis edáfico continuo.

In [76]:
# En Riego, dos valores poseen diferencias sutiles pero significativas a nivel de categorización:
# "No indica" ≠ "No Indica" # sus string son diferentes aunque signifiquen lo mismo
# mayúsculas, tildes, espacios, crean categorías nuevas
# sustituir "No Indica" por "No indica" en la columna "Riego"
data["Riego"] = data["Riego"].replace("No Indica", "No indica")

# En Estado, dos valores poseen diferencias sutiles pero significativas a nivel de categorización:
# eliminar espacios al inicio y final, y que el estring inicie en mayúscula y luego resto en minúscula

# 1. Eliminar espacios fantasmas al principio y final
data["Estado"] = data["Estado"].str.strip()
# 2. Unificar a formato Título (Primera de cada palabra en Mayúscula)
# Es más seguro que capitalize si tienes nombres compuestos como "En Producción"
data["Estado"] = data["Estado"].str.title() 
# 3.  RECONVERTIR A TIPO CATEGORY
data["Estado"] = data["Estado"].astype('category')


# En Fertilizantes aplicados, se unifican categorías similares
data["Fertilizantes aplicados"] = (
    data["Fertilizantes aplicados"]
    .str.strip()        # elimina espacios al inicio y final
    .str.lower()        # todo en minúsculas
)
data["Riego"] = data["Riego"].replace("ninguno", "no")
data["Fertilizantes aplicados"] = data["Fertilizantes aplicados"].astype('category')



In [77]:
# Análisis por columnas categóricas - Cardinalidad y valores únicos por categoría
for col in data.select_dtypes(include=['category']).columns:
    print(f"\nColumna: {col}")
    print(f"Cardinalidad: {data[col].nunique()}")
    print("Valores únicos:")
    for val in data[col].cat.categories:
        print(f" - {val}") # Mostrar cada valor único en la categoría


Columna: Departamento
Cardinalidad: 1
Valores únicos:
 - CUNDINAMARCA

Columna: Municipio
Cardinalidad: 1
Valores únicos:
 - FUNZA

Columna: Cultivo
Cardinalidad: 27
Valores únicos:
 - Ajo
 - Alfalfa
 - Caducifolios
 - Café
 - Calas
 - Clavel
 - Fresa
 - Granadilla
 - Guisantes
 - Gulupa
 - Hipérico
 - Hortalizas Varias
 - Lechuga
 - Maracuyá
 - Maíz
 - No Indica
 - Papa de año
 - Pasto
 - Pasto Kikuyo
 - Pasto KinGrass
 - Pastos
 - Quinua
 - Ruscus
 - Tomate
 - Tomate de Arbol
 - Uchuva
 - Zanahoria

Columna: Estado
Cardinalidad: 3
Valores únicos:
 - Establecido
 - No Indica
 - Por Establecer

Columna: Topografia
Cardinalidad: 7
Valores únicos:
 - Moderadamente ondulado
 - No indica
 - Ondulado
 - Pendiente
 - Pendiente fuerte
 - Pendiente moderada
 - Plano

Columna: Drenaje
Cardinalidad: 4
Valores únicos:
 - Buen drenaje
 - Mal drenaje
 - No indica
 - Regular drenaje

Columna: Riego
Cardinalidad: 6
Valores únicos:
 - Aspersión
 - Cañon
 - Goteo
 - Manguera
 - No Tiene
 - No indica



In [ ]:
# Exportar como cln_vista_agrosavia_Municipio_Funza_20260111_cleaned.csv
# data.to_csv("cln_vista_agrosavia_Municipio_Funza_20260111_cleaned.csv", index=False)